# Phase 2A — Building3D cache and label preparation

Run this notebook once with the Phase 1B output and the Phase 2 code bundle attached as Kaggle inputs.

In [ ]:
from pathlib import Path
import json
import shutil
import sys
import pandas as pd

code_file=next(Path("/kaggle/input").rglob("phase2_core.py"))
shutil.copy2(code_file,"/kaggle/working/phase2_core.py")
sys.path.insert(0,"/kaggle/working")
from phase2_core import *
seed_everything(42)
{"core":str(code_file),"torch":torch.__version__,"cuda":torch.cuda.is_available()}

In [ ]:
input_roots=[]
for path in Path("/kaggle/input").iterdir():
    if path.is_dir() and any(path.rglob("*.zip")):
        input_roots.append(path)
assert input_roots
source_root=max(input_roots,key=lambda x:sum(p.stat().st_size for p in x.rglob("*.zip")))
archive_table=archive_inventory(source_root)
manifest_path,manifest=find_split_manifest(source_root)
archive_table,manifest_path,manifest.split.value_counts()

In [ ]:
point_count=2048
max_corners=64
max_edges=128
shard_size=256
cache_root=Path("/kaggle/working/phase2_cache")
archive_table,index,rejected,cache_config=build_cache(source_root,cache_root,point_count,max_corners,max_edges,shard_size,42)
cache_config

In [ ]:
assert len(index)>0
assert set(index.split)=={"train","val","test"}
assert not index.building_id.duplicated().any()
assert index.shard.map(lambda x:(cache_root/x).exists()).all()
index.groupby("split").agg(samples=("building_id","size"),median_scale=("scale","median"),min_scale=("scale","min"),max_scale=("scale","max"))

In [ ]:
sample_set=RoofDataset(cache_root,"train")
sample=sample_set[0]
assert sample["points"].shape==(2048,8)
assert torch.isfinite(sample["points"]).all()
assert torch.isfinite(sample["corners"][:sample["corner_count"]]).all()
assert sample["edges"][:sample["edge_count"]].min()>=0
assert sample["edges"][:sample["edge_count"]].max()<sample["corner_count"]
{k:(tuple(v.shape) if torch.is_tensor(v) else v) for k,v in sample.items()}

In [ ]:
summary={"cache":cache_config,"index_rows":len(index),"rejected_rows":len(rejected),"split_counts":index.split.value_counts().to_dict(),"point_shape":list(sample["points"].shape),"corner_limit":max_corners,"edge_limit":max_edges}
Path("/kaggle/working/phase2_cache_summary.json").write_text(json.dumps(summary,indent=2))
summary